# Member 5 — Feature Engineering

**Technique:** Engineer handcrafted visual features from each leaf image (RGB + HSV histograms and channel statistics).

## Why this dataset needs it
Raw pixels are high-dimensional and noisy. Compact color descriptors summarise **health-related appearance** (greenness, discoloration) in a form classical models can learn from.


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Resolve Group_Deliverable root whether cwd is notebooks/ or deliverable root
HERE = Path.cwd().resolve()
ROOT = None
for p in (HERE, *HERE.parents):
    if (p / "src" / "preprocess_utils.py").exists():
        ROOT = p
        break
    if (p / "Group_Deliverable" / "src" / "preprocess_utils.py").exists():
        ROOT = p / "Group_Deliverable"
        break
if ROOT is None:
    raise FileNotFoundError("Run from progress/Group_Deliverable or its notebooks/ folder.")

sys.path.insert(0, str(ROOT / "src"))
from preprocess_utils import (
    SEED, SOURCE_URL, DOI, FOLDERS, CLASSES, paths,
    inventory_table, discover_images, audit_images,
    remove_exact_duplicates, iqr_mask, extract_feature_matrix, read_rgb,
    stratified_sample,
)

P = paths(ROOT)
RAW, VIZ, OUT, LOGS = P["raw"], P["viz"], P["outputs"], P["logs"]
for d in (VIZ, OUT, LOGS):
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(SEED)
print("Deliverable root:", ROOT)
print("Raw data:", RAW)
print("Dataset:", SOURCE_URL, "| DOI:", DOI)


In [ ]:
from skimage.color import rgb2hsv
from PIL import Image

meta_path = OUT / "m3_cleaned_no_outliers.csv"
meta = pd.read_csv(meta_path) if meta_path.exists() else audit_images(RAW)[0]
sample = stratified_sample(meta, 100)

# Build a readable feature table (subset of engineered columns)
rows = []
for row in sample.to_dict("records"):
    im = read_rgb(RAW / row["path"])
    rgb = np.asarray(im.resize((64, 64), Image.Resampling.BILINEAR), dtype=np.float32) / 255.0
    hsv = rgb2hsv(rgb)
    rows.append({
        "path": row["path"],
        "label": row["label"],
        "rgb_r_mean": float(rgb[:, :, 0].mean()),
        "rgb_g_mean": float(rgb[:, :, 1].mean()),
        "rgb_b_mean": float(rgb[:, :, 2].mean()),
        "hsv_h_mean": float(hsv[:, :, 0].mean()),
        "hsv_s_mean": float(hsv[:, :, 1].mean()),
        "hsv_v_mean": float(hsv[:, :, 2].mean()),
        "rgb_g_std": float(rgb[:, :, 1].std()),
        "hsv_s_std": float(hsv[:, :, 1].std()),
    })

feat_df = pd.DataFrame(rows)
display(feat_df.head())
feat_df.to_csv(OUT / "m5_engineered_color_features.csv", index=False)
print("Engineered feature rows:", len(feat_df))


## EDA visualization — correlation heatmap of engineered features


In [ ]:
num_cols = [c for c in feat_df.columns if c not in {"path", "label"}]
corr = feat_df[num_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation of engineered color features")
fig.tight_layout()
fig.savefig(VIZ / "m5_feature_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

# Class-wise greenness comparison
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=feat_df, x="label", y="rgb_g_mean", hue="label", order=CLASSES, palette=["#2ca02c", "#d62728"], ax=ax, legend=False)
ax.set_title("Mean green channel by class (engineered feature)")
fig.tight_layout()
fig.savefig(VIZ / "m5_greenness_by_class.png", dpi=150, bbox_inches="tight")
plt.show()
print("Interpretation: correlation shows redundancy among features; green-channel stats often separate healthy vs unhealthy leaves.")


## Viva talking points
1. Why engineer RGB/HSV summaries instead of raw pixels.
2. Walk through one feature (`rgb_g_mean`) and its meaning.
3. Interpret the correlation heatmap / greenness boxplot.
